# Session 12: Practice Quiz

**Notebook 2 of 2: Self-check** · about 30 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded. No cell calls an LLM or the network.

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


def chain(schema, *nodes):
    """A graph that runs the given (name, function) nodes in order."""
    builder = StateGraph(schema)
    previous = START
    for name, function in nodes:
        builder.add_node(name, function)
        builder.add_edge(previous, name)
        previous = name
    builder.add_edge(previous, END)
    return builder.compile()


print("ready")

## Q1 · Predict: three keys, two reducers

```python
class State(TypedDict, total=False):
    total: int
    count: Annotated[int, operator.add]
    log: Annotated[list[str], operator.add]
```

Node `a` returns `{"total": 1, "count": 1, "log": ["a"]}`, then node `b` returns `{"total": 2, "count": 2, "log": ["b"]}`. The graph is invoked with `{"total": 10, "count": 10, "log": ["start"]}`. What is the final state?

In [ ]:
class State(TypedDict, total=False):
    total: int
    count: Annotated[int, operator.add]
    log: Annotated[list[str], operator.add]


graph = chain(
    State,
    ("a", lambda state: {"total": 1, "count": 1, "log": ["a"]}),
    ("b", lambda state: {"total": 2, "count": 2, "log": ["b"]}),
)
print(graph.invoke({"total": 10, "count": 10, "log": ["start"]}))

<details>
<summary>Show answer</summary>

`{'total': 2, 'count': 13, 'log': ['start', 'a', 'b']}`

- `total` has no reducer, so each update **replaces** it: the last writer, `b`, wins with `2`.
- `count`'s reducer is `operator.add` on integers: `10 + 1 + 2 = 13`. A reducer is any two-argument function, so it can sum, append, merge dicts or keep a maximum.
- `log`'s reducer is `operator.add` on lists, which concatenates. The input value is the starting point, so `"start"` stays first.

The reducer is applied to the **input** too: the initial state is the first value, and every update is combined with it.

</details>

## Q2 · Predict: how many items does `stream` yield?

For the two-node graph from Q1, how many items does `graph.stream(input, stream_mode="updates")` yield? And `stream_mode="values"`? Why do they differ?

In [ ]:
start = {"total": 0, "count": 0, "log": []}
updates = list(graph.stream(start, stream_mode="updates"))
values = list(graph.stream(start, stream_mode="values"))
print(len(updates), updates)
print(len(values), values)

<details>
<summary>Show answer</summary>

**2 updates and 3 values.**

- `"updates"` yields one item per node that ran, `{node_name: what it returned}`: one for `a`, one for `b`. It shows *who changed what*, and nothing about keys that weren't touched.
- `"values"` yields the **whole state** at each point: first the input, then after `a`, then after `b`. It shows *what the state looked like* at each step.

When you debug a wrong route, start with `"updates"` to see which node produced the bad value, then look at the `"values"` item just before it to see what that node received.

</details>

## Q3 · Debug: the counter that crashes

This graph should count its attempts. It crashes on its first step. Find the bug before running the cell, and give two ways to fix it.

```python
class RetryState(TypedDict, total=False):
    attempts: int


def attempt(state: RetryState) -> dict:
    return {"attempts": state["attempts"] + 1}

graph.invoke({})
```

In [ ]:
class RetryState(TypedDict, total=False):
    attempts: int


def attempt(state: RetryState) -> dict:
    return {"attempts": state["attempts"] + 1}


try:
    chain(RetryState, ("attempt", attempt)).invoke({})
except KeyError as error:
    print("KeyError:", error)

<details>
<summary>Show answer</summary>

`total=False` makes `attempts` **optional**: until something sets it, it isn't in the state at all, so `state["attempts"]` raises `KeyError: 'attempts'`. The traceback points at the node's line, which is the useful part.

Two fixes:

- **Default in the node:** `state.get("attempts", 0) + 1`. The node no longer depends on how it was started.
- **Initialise it in the input:** `graph.invoke({"attempts": 0})`. That makes the starting state explicit; a wrapper such as the workbook's `GraphRouter.route` is the natural place.

Doing both is reasonable: the wrapper documents the starting state, and the node is robust when it's tested on its own.

</details>

## Q4 · Engineering judgment: graph or function?

A colleague proposes LangGraph for a new feature: *"Read a PDF, summarise it with the LLM, save the summary to the database. Three nodes, three edges. That way it's extensible."*

What would you recommend, and what would change your mind?

<details>
<summary>Show answer</summary>

**Plain functions.** The workflow is a straight line: one path, no loop, no decision, nothing to wait for. As a graph it needs a state schema, three nodes, edges and a compile step to express what three function calls express directly, and its tracebacks run through LangGraph's code. "Extensible" isn't a requirement; adding a step to a function is easy too.

```python
def summarise_document(path: Path) -> Summary:
    text = extract_text(path)
    summary = summariser.summarise(text)
    repository.save(summary)
    return summary
```

What would change the recommendation:

- The summary must be **approved by a person** before it's saved, possibly hours later: pausing and resuming with a checkpointer.
- **Long documents** must survive restarts mid-way, resuming from the last finished step.
- The flow gains real **branches and loops**: retry the summary with feedback when it fails validation, route by document type, escalate.
- People need to **watch progress** step by step.

Keep `extract_text`, `summarise` and `save` as plain functions either way. If a graph is needed later, they become its nodes' bodies.

</details>

## Q5 · Predict: the smallest recursion limit

A graph runs `validate`, which loops back to itself while the input is invalid and `attempts < 3`, then goes to `publish` or `escalate`, then ends. What's the longest path, and what's the smallest `recursion_limit` that lets it finish? Run the cell to check.

In [ ]:
class ValidateState(TypedDict, total=False):
    attempts: int
    steps: Annotated[list[str], operator.add]


def validate(state: ValidateState) -> dict:
    return {"attempts": state.get("attempts", 0) + 1, "steps": ["validate"]}


def after_validate(state: ValidateState) -> str:
    still_invalid = True  # the worst case: invalid every time
    if still_invalid and state["attempts"] < 3:
        return "validate"
    return "escalate" if still_invalid else "publish"


builder = StateGraph(ValidateState)
builder.add_node("validate", validate)
builder.add_node("publish", lambda state: {"steps": ["publish"]})
builder.add_node("escalate", lambda state: {"steps": ["escalate"]})
builder.add_edge(START, "validate")
builder.add_conditional_edges("validate", after_validate, ["validate", "publish", "escalate"])
builder.add_edge("publish", END)
builder.add_edge("escalate", END)
graph = builder.compile()

for limit in range(1, 7):
    try:
        final = graph.invoke({"steps": []}, {"recursion_limit": limit})
        print(f"limit {limit}: finished, {final['steps']}")
        break
    except GraphRecursionError:
        print(f"limit {limit}: GraphRecursionError")

<details>
<summary>Show answer</summary>

The longest path is `validate` three times, then `escalate`: **4 nodes**. The smallest limit that lets it finish is **5**, one more than the number of nodes, as the concepts notebook measured.

The practical lesson isn't the "+1": it's to **derive the limit from your longest legitimate path, and test it**. Then a routing bug fails after a handful of steps instead of after LangGraph's default (10,007 steps in the version this course uses), which, with an LLM call in the loop, is 10,007 paid requests.

</details>

## Q6 · Code completion: a routing function

Routing functions are ordinary functions of the state, so you can write and test them without a graph. Complete `after_review` so that it returns:

- `"revise"` when there are `issues` and `revisions` is below 2;
- `"escalate"` when there are `issues` and `revisions` is 2 or more;
- `"publish"` when there are no issues.

`issues` may be missing from the state; treat that as no issues.

In [ ]:
from typing import Literal


def after_review(state: dict) -> Literal["revise", "escalate", "publish"]:
    # TODO
    return "publish"


cases = [
    ({"issues": ["too long"], "revisions": 0}, "revise"),
    ({"issues": ["too long"], "revisions": 2}, "escalate"),
    ({"issues": [], "revisions": 1}, "publish"),
    ({"revisions": 0}, "publish"),
]
for state, expected in cases:
    result = after_review(state)
    print("ok  " if result == expected else "FAIL", state, "->", result)

<details>
<summary>Show answer</summary>

```python
def after_review(state: dict) -> Literal["revise", "escalate", "publish"]:
    if not state.get("issues"):
        return "publish"
    if state.get("revisions", 0) < 2:
        return "revise"
    return "escalate"
```

- Handling "no issues" first keeps the other two branches simple.
- `state.get(...)` with a default covers keys that aren't set yet, as Q3 showed.
- The `Literal` return type documents every destination, and LangGraph reads it to draw the edges when you call `add_conditional_edges("review", after_review)` without a list.
- Testing it with plain dicts, as the cell does, is the cheapest test a graph can have. Do this for every routing function.

</details>

## Q7 · Engineering judgment: catching everything in a node

A reviewer finds this node in a workflow that triages reports with an LLM:

```python
def triage(state):
    try:
        return {"triage": triager.triage(state["report"]), "status": "ok"}
    except Exception as error:
        return {"status": "failed", "error": str(error)}
```

`"failed"` routes to the keyword-rules fallback. What's the risk, and what would you change?

<details>
<summary>Show answer</summary>

`except Exception` turns **every** problem into an ordinary route: a rejected API key, a misspelled model name, a bug such as an `AttributeError` in the triager, even a `KeyError` from a missing `report`. All of them quietly become "use the keyword rules". The run reports success, every report is triaged by keywords, and nobody notices that the LLM hasn't been called for days.

Instead:

- Catch only the failures the workflow **knows how to handle**, and treat them differently: invalid output, retry; provider unavailable, fall back; refusal, review.
- Let **configuration errors** propagate and stop the run, because every later report would fail the same way (Session 09's `LLMConfigurationError`).
- Let **bugs** propagate. LangGraph passes exceptions from nodes through unchanged, so your tests and logs see them.
- Record the error in the state for the cases you do handle, so the result shows *why* the fallback was used. The workbook's `Routing.error` does this.

</details>

## Q8 · Predict: what runs twice?

This node counts how often its code runs, before and after `interrupt`. The graph is invoked once (it pauses), then resumed with `"yes"`. What's in `runs` at the end?

In [ ]:
runs: list[str] = []


class ApprovalState(TypedDict, total=False):
    decision: str


def approve(state: ApprovalState) -> dict:
    runs.append("before interrupt")
    answer = interrupt("Approve the page?")
    runs.append("after interrupt")
    return {"decision": answer}


builder = StateGraph(ApprovalState)
builder.add_node("approve", approve)
builder.add_edge(START, "approve")
builder.add_edge("approve", END)
graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "quiz-1"}}

graph.invoke({}, config)
print("after the first invoke:", runs)
print(graph.invoke(Command(resume="yes"), config))
print("after resuming:        ", runs)

<details>
<summary>Show answer</summary>

`['before interrupt', 'before interrupt', 'after interrupt']`

When a run resumes, the interrupted node runs again **from its first line**. This time `interrupt(...)` returns the resume value instead of pausing. So everything before `interrupt` runs twice: once before the pause and once after it.

That matters when the code before `interrupt` has side effects: sending a notification, calling an LLM, writing to a database. Either put the side effect in an **earlier node**, whose result is already saved in the checkpoint, or make it safe to repeat (idempotent). The same rule applies to durable workflow engines such as Temporal or Step Functions: a resumed step may run again.

</details>

## Q9 · Comparison: testing a graph

You've built the workbook's routing workflow twice, as `PlainRouter` and `GraphRouter`. List the kinds of tests you would keep for the graph version, cheapest first. Which one would catch each of these bugs?

1. The rule "security incident below P2" uses `P3` only, and forgets `P4`.
2. `check` is wired to `queue` and `review`, but `page` was never declared as a destination.
3. The graph retries an unavailable provider, while the plain version doesn't.
4. A routing bug loops `triage` forever.

<details>
<summary>Show answer</summary>

**The layers, cheapest first:**

1. **Business rules and routing functions as plain functions**: called with values or dicts, no graph.
2. **The graph through `invoke`**, with scripted fake triagers: assert on the final state and the path.
3. **The structure**: node names and edges from `get_graph()`.
4. **A contract with the plain version**: the same tests run against both routers.
5. **The safety net**: the recursion limit is set, derived from the longest path, and passed to every run.

**Which test catches which bug:**

1. The P4 rule: a **rule test** (`review_reasons` with a security P4 triage). It's a plain function, so the cheapest test finds it.
2. The undeclared `page`: a **structure test** on the edges. The graph may still run correctly, so path tests alone might miss it; the drawing would be wrong.
3. The extra retry: the **contract**. The same "unavailable provider" test expects one attempt and a fallback, and fails only for the graph.
4. The infinite loop: the **recursion limit test**. Without a tight limit, the loop would run 10,007 steps before failing; with one, it fails after a few.

</details>

---

**Done?** Go back over any question you got wrong; each relates to a part of `01-concepts.ipynb`. Then continue **Project 03** with the [workbook](../workbook/assignment.md).